# Teresa's view of `curated-orders`

The orders Dan curates with Flink, read as an **Iceberg table** that Tableflow keeps up to date in Confluent-managed storage. This notebook only reads: it creates nothing in Confluent Cloud, and it takes every value from the shared `.env` (see `README.md`).


In [ ]:
# Settings come from the environment that `make teresa-notebook` loaded from .env.
from catalog_config import MissingConfig, friendly_message, load_settings

try:
    settings = load_settings()
    print(f"Reading {'.'.join(settings.table_identifier)} through the {settings.catalog_name} catalog")
except MissingConfig as error:
    settings = None
    print(error)


In [ ]:
# Open the Tableflow Iceberg REST catalog and load the table.
table = None
if settings is not None:
    from pyiceberg.catalog import load_catalog
    from pyiceberg.exceptions import NoSuchNamespaceError, NoSuchTableError

    try:
        catalog = load_catalog(settings.catalog_name, **settings.properties)
        table = catalog.load_table(settings.table_identifier)
    except (NoSuchNamespaceError, NoSuchTableError):
        print(friendly_message("missing"))
    except Exception as error:
        # A wrong endpoint or warehouse lands here; the README explains the two overrides.
        print(f"Could not open the catalog ({type(error).__name__}). See the README for TABLEFLOW_CATALOG_ENDPOINT and TABLEFLOW_CATALOG_WAREHOUSE.")
        raise


In [ ]:
# Read the table into pandas.
orders = None
if table is not None:
    orders = table.scan().to_pandas()
    if orders.empty:
        print(friendly_message("empty"))
        orders = None
    else:
        print(f"{len(orders)} orders")


In [ ]:
# Orders by state.
if orders is not None:
    ax = orders["state"].value_counts().sort_values().plot.barh(figsize=(8, 5))
    ax.set_xlabel("orders")
    ax.set_title("Orders by state")


In [ ]:
# The ten most recent orders.
if orders is not None:
    from IPython.display import display

    # Inside an if block Jupyter would not show the last expression, so display it explicitly.
    display(orders.sort_values("ordertime", ascending=False).head(10))
